# XLS-R 300M and w2v-BERT 2.0 — baselines

Kaggle writes `/kaggle/working` to a version only when a commit completes, so
every improved checkpoint is mirrored to a private Hugging Face repo as it is
produced and pulled back on restart. Local checkpoints are written every 100
steps.


In [ ]:
!pip -q install "transformers>=4.44" soundfile accelerate huggingface_hub 2>&1 | tail -1
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
import os, glob, json, subprocess, tarfile, shutil
WORK='/kaggle/working'; os.chdir(WORK)

# HF token: the single thing that makes this survive Kaggle dying.
try:
    from kaggle_secrets import UserSecretsClient
    os.environ['HF_TOKEN'] = UserSecretsClient().get_secret("HF_TOKEN").strip()
    print('HF_TOKEN loaded — checkpoints will be mirrored off Kaggle')
except Exception as e:
    os.environ['HF_TOKEN'] = ''
    print('*** NO HF_TOKEN SECRET. Checkpoints live only in this container ***')
    print('*** If it dies, everything is lost. Add-ons -> Secrets -> HF_TOKEN ***')

HF_USER = 'rmaacario'          # <-- change if your HF username differs

# Three ways the data can arrive, in order of preference:
#   1. attached and already extracted  -> validated.tsv is visible
#   2. attached as a .tar.gz           -> extract it to /kaggle/temp (free quota)
#   3. not attached at all             -> download (slow; warn loudly)
hits = glob.glob('/kaggle/input/**/validated.tsv', recursive=True)
tars = glob.glob('/kaggle/input/**/*scripted*.tar.gz', recursive=True) \
     + glob.glob('/kaggle/input/**/cv-corpus*.tar.gz', recursive=True)
if hits:
    QXP = os.path.dirname(hits[0]); print('attached input (extracted):', QXP)
elif tars:
    dest = '/kaggle/temp/data' if os.path.isdir('/kaggle/temp') else f'{WORK}/data'
    os.makedirs(dest, exist_ok=True)
    print(f'attached archive {tars[0]} — extracting to {dest} (a few minutes)')
    with tarfile.open(tars[0]) as t: t.extractall(dest)
    f = glob.glob(f'{dest}/**/validated.tsv', recursive=True)
    if not f: raise SystemExit(f'archive extracted but no validated.tsv under {dest}')
    QXP = os.path.dirname(f[0]); print('extracted to:', QXP)
else:
    print('*** NO INPUT ATTACHED — downloading (~40 min). Stop and attach it. ***')
    RAW=f'{WORK}/raw'; os.makedirs(RAW, exist_ok=True)
    key = UserSecretsClient().get_secret("MDC_API_KEY").strip()
    for ds_id,name in [("cmqio1wd400xynq079ifamac6","scripted"),
                       ("cmqialqsu007bnr07e4se1rl8","spontaneous")]:
        out=f'{RAW}/{name}.tar.gz'
        if os.path.exists(out): print(name,'present'); continue
        r=subprocess.run(["curl","-sS","-X","POST",
            f"https://mozilladatacollective.com/api/datasets/{ds_id}/download",
            "-H",f"Authorization: Bearer {key}","-H","Content-Type: application/json"],
            capture_output=True,text=True)
        j=json.loads(r.stdout)
        if "downloadUrl" not in j: print(name,"FAILED:",r.stdout[:200]); continue
        subprocess.run(["curl","-fL","-o",out,j["downloadUrl"]],check=True)
        with tarfile.open(out) as t: t.extractall(RAW)
    f=glob.glob(f'{RAW}/**/validated.tsv',recursive=True)
    if not f: raise SystemExit('no data — attach it as a Kaggle Dataset')
    QXP=os.path.dirname(f[0])

# Kaggle can shard an uploaded archive across many numbered directories, so
# clips/ is NOT reliably a sibling of validated.tsv. Verify, and if it is
# missing, build a flat symlink farm in /kaggle/temp over every mp3 found
# anywhere under the inputs. Symlinks cost no disk and no quota.
CLIPS = f'{QXP}/clips'
if not (os.path.isdir(CLIPS) and len(os.listdir(CLIPS)) > 1000):
    print(f'{CLIPS} missing or sparse — indexing mp3s across all inputs...')
    roots = ['/kaggle/input', f'{WORK}/raw',
             '/kaggle/temp/data' if os.path.isdir('/kaggle/temp') else '']
    mp3s = []
    for r in roots:
        if r and os.path.isdir(r):
            mp3s += glob.glob(f'{r}/**/*.mp3', recursive=True)
    if not mp3s:
        raise SystemExit('no .mp3 found anywhere under the attached inputs')
    CLIPS = '/kaggle/temp/clips' if os.path.isdir('/kaggle/temp') else f'{WORK}/clips_idx'
    os.makedirs(CLIPS, exist_ok=True)
    made = 0
    for p in mp3s:
        d = os.path.join(CLIPS, os.path.basename(p))
        if not os.path.lexists(d):
            os.symlink(p, d); made += 1
    print(f'indexed {len(mp3s)} mp3s ({made} new links) -> {CLIPS}')
print('clips:', len(os.listdir(CLIPS)))
print(f'free: {shutil.disk_usage(WORK).free/1e9:.1f} GB')

In [ ]:
SCRIPTS={}
SCRIPTS['make_split_grouped.py']=r'''#!/usr/bin/env python3
"""Build splits that are disjoint by SENTENCE, not just by speaker.

The organisers' data/make_split.py groups by client_id and asserts speaker
disjointness; it never mentions the sentence field. Because the scripted
corpus is ~2,070 prompts recorded ~11 times each, a speaker-wise split leaves
essentially every test sentence in the training text: measured overlap on the
released manifests is 2298/2299 clips (99.96%) for scripted test and 85% for
valid. Whatever those splits measure, it is not recognition of unseen text.

A caveat this script checks rather than assumes: clips are linked both by
speaker and by sentence, so "disjoint on both axes" may be impossible. If many
speakers read the same prompts, the speaker-sentence graph collapses into one
connected component and no split can separate both. We report the component
structure first, then split on sentence, and report the speaker overlap that
results as a known, stated cost.
"""
import argparse, json, sys
import pandas as pd


def components(df):
    """Connected components of the clip graph linked by shared speaker or sentence."""
    parent = {}
    def find(x):
        parent.setdefault(x, x)
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x
    def union(a, b):
        ra, rb = find(a), find(b)
        if ra != rb:
            parent[ra] = rb
    for _, r in df.iterrows():
        union(("spk", r.client_id), ("sent", r.sentence))
    lab = df.apply(lambda r: find(("spk", r.client_id)), axis=1)
    return lab


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--input", required=True, help="Common Voice TSV (needs client_id, sentence)")
    ap.add_argument("--output-dir", default="data/splits_grouped")
    ap.add_argument("--dev-ratio", type=float, default=0.10)
    ap.add_argument("--test-ratio", type=float, default=0.10)
    ap.add_argument("--seed", type=int, default=1)
    a = ap.parse_args()

    df = pd.read_csv(a.input, sep="\t")
    for c in ("client_id", "sentence"):
        if c not in df.columns:
            sys.exit(f"missing required column: {c}  (have: {list(df.columns)})")
    df["sentence"] = df["sentence"].astype(str).str.strip()

    print(f"{len(df)} clips  {df.client_id.nunique()} speakers  "
          f"{df.sentence.nunique()} unique sentences")
    print(f"mean recordings per sentence: {len(df)/df.sentence.nunique():.1f}")

    lab = components(df)
    sizes = lab.value_counts()
    print(f"\nspeaker+sentence connected components: {len(sizes)}")
    print(f"largest holds {sizes.iloc[0]} clips ({100*sizes.iloc[0]/len(df):.1f}% of corpus)")
    if sizes.iloc[0] > 0.5 * len(df):
        print("=> One component dominates: splits CANNOT be disjoint on both axes.\n"
              "   Splitting on SENTENCE, since generalising to unseen text is what\n"
              "   the held-out test set will actually require. Speaker overlap is\n"
              "   reported below as the accepted cost.")

    # Sentence-wise assignment, greedy on clip count to hit the target ratios.
    rng = pd.Series(sorted(df.sentence.unique())).sample(frac=1.0, random_state=a.seed)
    counts = df.sentence.value_counts()
    n = len(df)
    targets = {"test": a.test_ratio * n, "dev": a.dev_ratio * n}
    got = {"test": 0, "dev": 0}
    assign = {}
    for s in rng:
        for split in ("test", "dev"):
            if got[split] < targets[split]:
                assign[s] = split
                got[split] += counts[s]
                break
        else:
            assign[s] = "train"
    df["split"] = df.sentence.map(assign)

    print("\n--- resulting splits ---")
    for split in ("train", "dev", "test"):
        d = df[df.split == split]
        print(f"{split:6s} {len(d):6d} clips  {d.sentence.nunique():5d} sentences  "
              f"{d.client_id.nunique():4d} speakers")

    tr = set(df[df.split == "train"].sentence)
    for split in ("dev", "test"):
        d = df[df.split == split]
        leak = d.sentence.isin(tr).sum()
        print(f"sentence leakage {split}: {leak}/{len(d)} ({100*leak/max(len(d),1):.2f}%)")
    trs = set(df[df.split == "train"].client_id)
    for split in ("dev", "test"):
        d = df[df.split == split]
        ov = d.client_id.isin(trs).sum()
        print(f"speaker overlap  {split}: {ov}/{len(d)} ({100*ov/max(len(d),1):.1f}%)  [accepted]")

    import os
    os.makedirs(a.output_dir, exist_ok=True)
    for split in ("train", "dev", "test"):
        p = f"{a.output_dir}/{split}.tsv"
        df[df.split == split].drop(columns=["split"]).to_csv(p, sep="\t", index=False)
        print(f"wrote {p}")


if __name__ == "__main__":
    main()
'''
SCRIPTS['train_qxp.py']=r'''"""Fine-tune XLS-R 300M on Puno Quechua, on splits that are actually held out.

**Why this exists rather than the organisers' recipe.** Their fine-tune is
fairseq-hydra on a corpus split by `client_id` alone (data/make_split.py, which
asserts speaker disjointness and never mentions the sentence field). The
scripted corpus is 2,067 prompts recorded ~11 times each, so that split leaves
2298 of 2299 test clips with their exact sentence present in the training text.
Whatever their 1.19% WER measures, it is not transcription of unseen speech.
This script trains on data/splits_grouped/, which is sentence-disjoint.

**Speaker overlap is accepted and must be reported.** The speaker+sentence clip
graph is one connected component covering the whole corpus, so no split can be
disjoint on both axes. Sentence-disjointness is the one that matters, because
the held-out test set requires generalising to unseen text. Common Voice's own
train/dev/test are clean on both axes and are the stricter benchmark, at the
cost of 709 training clips; run both and report both.

**Full fine-tune, not adapters.** The Kadiwéu pipeline trains a ~2M-parameter
MMS adapter because 25 minutes of audio cannot support more. Here there are
~30 hours, XLS-R has no per-language adapters to init, and the organisers'
own ablation shows the encoder needs to move. The convolutional feature
encoder still stays frozen: its representations come from self-supervised
pretraining on far more speech than Quechua has, and rewriting them is how
fine-tuning collapses.

**Character CTC.** The output inventory is the characters of the training text,
built here rather than hardcoded. The apostrophe is phonemic — it marks
ejectives, 12,225 occurrences — and ⟨ñ⟩ is a distinct phoneme, so neither may
be folded away. Nothing is lowercased beyond what the official scorer does
(lowercase, then strip ? ! ¿ ¡ . ,), because normalising training targets
against unnormalised references turns every rewrite into a guaranteed error.

    python train_qxp.py --splits data/splits_grouped --epochs 8
"""

import argparse
import json
import os
import random
import re
import time

# The scorer the organisers use, reproduced exactly (eval/omnilingual/compute_wer.py):
#   "".join(c for c in text.lower() if c not in "?!¿¡.,").strip()
# Training targets go through this and nothing else, so that what the model
# learns to emit is what the references are scored as.
SCORE_STRIP = set("?!¿¡.,")


def normalise(text):
    return "".join(c for c in str(text).lower() if c not in SCORE_STRIP).strip()


def set_all_seeds(seed):
    import numpy as np
    import torch
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.use_deterministic_algorithms(False)  # MPS lacks deterministic kernels


def build_vocab(texts, path):
    chars = sorted({c for t in texts for c in t if c != " "})
    vocab = {"[PAD]": 0, "[UNK]": 1, "|": 2}
    for c in chars:
        vocab.setdefault(c, len(vocab))
    json.dump(vocab, open(path, "w", encoding="utf-8"), ensure_ascii=False, indent=1)
    print(f"vocab: {len(vocab)} symbols -> {path}")
    return vocab


def cer(ref, hyp):
    r, h = list(ref), list(hyp)
    prev = list(range(len(h) + 1))
    for i, rc in enumerate(r, 1):
        cur = [i]
        for j, hc in enumerate(h, 1):
            cur.append(min(prev[j] + 1, cur[j - 1] + 1, prev[j - 1] + (rc != hc)))
        prev = cur
    return prev[-1], max(len(r), 1)


def wer(ref, hyp):
    r, h = ref.split(), hyp.split()
    prev = list(range(len(h) + 1))
    for i, rw in enumerate(r, 1):
        cur = [i]
        for j, hw in enumerate(h, 1):
            cur.append(min(prev[j] + 1, cur[j - 1] + 1, prev[j - 1] + (rw != hw)))
        prev = cur
    return prev[-1], max(len(r), 1)


_RS = {}


def read_audio(clips_dir, path, max_sec):
    """Common Voice ships MP3 at 32 kHz; wav2vec2 wants 16 kHz mono float.

    Decoding goes through soundfile rather than torchaudio.load: as of
    torchaudio 2.11 that call delegates to TorchCodec and raises without it,
    while libsndfile reads these MP3s directly. Resamplers are cached per
    source rate — building one per clip dominates data loading otherwise.
    """
    import os
    import numpy as np
    import soundfile as sf
    import torch
    import torchaudio
    y, sr = sf.read(os.path.join(clips_dir, path), dtype="float32")
    if y.ndim > 1:
        y = y.mean(1)
    y = torch.from_numpy(np.ascontiguousarray(y))
    if sr != 16000:
        if sr not in _RS:
            _RS[sr] = torchaudio.transforms.Resample(sr, 16000)
        y = _RS[sr](y)
    if len(y) > max_sec * 16000:
        y = y[: int(max_sec * 16000)]
    return y


class Utt:
    """Defined at module level on purpose — see read_audio's note on spawn."""

    def __init__(self, rows, clips_dir, vocab, max_sec, augment=False,
                 feature_extractor=None):
        self.rows, self.clips_dir = rows, clips_dir
        self.vocab, self.max_sec, self.augment = vocab, max_sec, augment
        # w2v-BERT consumes 80-bin mel filterbanks, not raw audio. Passing a
        # feature extractor switches this dataset to emit those instead.
        self.fe = feature_extractor

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, i):
        import numpy as np
        import torch
        r = self.rows[i]
        y = read_audio(self.clips_dir, r["path"], self.max_sec)
        if self.augment:
            # Speed perturbation only. Pitch shift would distort the vowel
            # qualities carrying the trivocalic/loanword contrast.
            f = random.choice([0.9, 1.0, 1.1])
            if f != 1.0:
                n = int(len(y) / f)
                y = torch.from_numpy(np.interp(
                    np.linspace(0, len(y) - 1, n), np.arange(len(y)),
                    y.numpy()).astype("float32"))
        # Spaces become the word-separator symbol, never dropped: without ⟨|⟩
        # the model emits an unbroken character string, CER inflates by every
        # missing space and WER pins at 100%.
        ids = [self.vocab["|"] if c == " "
               else self.vocab.get(c, self.vocab["[UNK]"]) for c in r["text"]]
        if self.fe is not None:
            y = self.fe(y.numpy(), sampling_rate=16000,
                        return_tensors="pt").input_features[0]
        return y, torch.tensor(ids), r["path"]


def collate(b):
    """Pad a batch. Raw-waveform models (XLS-R) get 1-D tensors; w2v-BERT gets
    2-D mel frames, so the padded tensor needs the feature axis."""
    import torch
    xs, ys, ids = zip(*b)
    xl = torch.tensor([len(x) for x in xs])
    yl = torch.tensor([len(y) for y in ys])
    if xs[0].dim() == 2:                       # (frames, mel_bins)
        X = torch.zeros(len(xs), int(max(xl)), xs[0].shape[1])
    else:                                      # (samples,)
        X = torch.zeros(len(xs), int(max(xl)))
    for i, x in enumerate(xs):
        X[i, : len(x)] = x
    return X, torch.cat(ys), xl, yl, ids



def save_ckpt(path, model, opt, sched, scaler, step, epoch, best, args):
    """Atomic checkpoint: everything needed to resume mid-epoch.

    Written to a temp path and renamed, because a session killed *during* a
    save must not leave a corrupt file that breaks the resume. Holds optimiser,
    scheduler, AMP scaler, step, epoch, best-so-far and RNG state — weights
    alone would silently restart the schedule and the data order.

    **Optimiser state is dropped above ~400M parameters.** AdamW keeps two
    fp32 tensors per parameter, so a 581M model's checkpoint is ~7 GB and the
    atomic rename needs 14 GB at peak — which overran Kaggle's 19.5 GB working
    directory and killed a w2v-BERT run. Without it the file is ~2.3 GB.
    Resume then restarts Adam's moments, costing a few steps of momentum;
    that is a far better trade than not being able to resume at all.

    Refuses to write at all below 3 GB free, because a half-written checkpoint
    plus a full disk is how you lose both the run and the ability to restart it.
    """
    import os, random, shutil, numpy as np, torch
    free_gb = shutil.disk_usage(os.path.dirname(path) or ".").free / 1e9
    if free_gb < 3.0:
        print(f"  !! only {free_gb:.1f} GB free — skipping checkpoint", flush=True)
        return
    n_par = sum(p.numel() for p in model.parameters())
    light = n_par > 400e6
    tmp = path + ".tmp"
    torch.save({
        "model": model.state_dict(),
        "opt": None if light else opt.state_dict(),
        "sched": sched.state_dict(),
        "scaler": scaler.state_dict(),
        "step": step, "epoch": epoch, "best": best,
        "args": vars(args),
        "rng": {"python": random.getstate(),
                "numpy": np.random.get_state(),
                "torch": torch.get_rng_state()},
    }, tmp)
    os.replace(tmp, path)


def load_ckpt(path, model, opt, sched, scaler):
    import os, random, numpy as np, torch
    if not os.path.exists(path):
        return 0, 0, 1e9
    try:
        c = torch.load(path, map_location="cpu", weights_only=False)
    except Exception as e:
        print(f"  checkpoint at {path} unreadable ({e}); starting fresh", flush=True)
        return 0, 0, 1e9
    model.load_state_dict(c["model"])
    if c.get("opt") is not None:
        opt.load_state_dict(c["opt"])
    else:
        print("  (checkpoint has no optimiser state — Adam moments restart)", flush=True)
    sched.load_state_dict(c["sched"])
    scaler.load_state_dict(c["scaler"])
    random.setstate(c["rng"]["python"])
    np.random.set_state(c["rng"]["numpy"])
    torch.set_rng_state(c["rng"]["torch"].cpu().to(torch.uint8))
    print(f"  RESUMED from {path}: epoch {c['epoch']}, step {c['step']}, "
          f"best dev CER {c['best']:.2f}", flush=True)
    return c["step"], c["epoch"], c["best"]



def hf_push(local_dir, repo, token, note=""):
    """Mirror a finished checkpoint to the Hugging Face Hub.

    This is the only protection that survives the container itself. Kaggle
    writes /kaggle/working to a version only when a commit *completes*; a run
    that hangs or is cancelled takes every local checkpoint with it, which is
    exactly how 10 GPU-hours were lost. Pushing out of the box means the run
    can die at any moment without costing the weights.

    Failures here are logged and swallowed: a network problem must never kill
    a training run that is otherwise fine.
    """
    if not repo or not token:
        return
    try:
        from huggingface_hub import HfApi
        api = HfApi(token=token)
        api.create_repo(repo, private=True, exist_ok=True)
        api.upload_folder(folder_path=local_dir, repo_id=repo,
                          commit_message=note or "checkpoint")
        print(f"  pushed {local_dir} -> hf.co/{repo} ({note})", flush=True)
    except Exception as e:
        print(f"  HF push failed (continuing): {type(e).__name__}: {e}", flush=True)


def hf_pull(repo, token, dest):
    """Restore a checkpoint pushed by a previous, dead session."""
    if not repo or not token:
        return False
    try:
        from huggingface_hub import snapshot_download
        import os
        if os.path.exists(os.path.join(dest, "config.json")):
            return False                      # local copy already present
        snapshot_download(repo_id=repo, token=token, local_dir=dest)
        print(f"  restored {repo} -> {dest}", flush=True)
        return True
    except Exception as e:
        print(f"  no HF checkpoint to restore ({type(e).__name__})", flush=True)
        return False



def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--splits", default="data/splits_grouped")
    ap.add_argument("--clips", default="data/raw/cv-corpus-26.0-2026-06-12/qxp/clips")
    ap.add_argument("--vocab", default="vocab_qxp.json")
    ap.add_argument("--model", default="facebook/wav2vec2-xls-r-300m")
    ap.add_argument("--seed", type=int, default=1)
    ap.add_argument("--epochs", type=int, default=8)
    ap.add_argument("--lr", type=float, default=5e-5)   # baseline paper, XLS-R-300M
    ap.add_argument("--batch", type=int, default=4)
    ap.add_argument("--accum", type=int, default=8)
    ap.add_argument("--warmup", type=float, default=0.1)
    ap.add_argument("--max-sec", type=float, default=20.0)
    ap.add_argument("--speed-perturb", action="store_true")
    ap.add_argument("--limit", type=int, default=0, help="debug: cap rows per split")
    ap.add_argument("--out", default="runs_qxp")
    ap.add_argument("--save-every", type=int, default=100,
                    help="checkpoint every N optimiser steps; 0 disables")
    ap.add_argument("--workers", type=int, default=2)
    ap.add_argument("--hf-repo", default="",
                    help="e.g. rmaacario/qxp-xlsr — mirrors best/ off Kaggle")
    ap.add_argument("--hf-token", default=os.environ.get("HF_TOKEN", ""))
    args = ap.parse_args()

    set_all_seeds(args.seed)
    import csv
    import numpy as np
    import torch
    import torchaudio
    import transformers

    # Two encoder families, chosen by --model:
    #   wav2vec2 / XLS-R  — consumes raw 16 kHz waveform, has a conv feature
    #                       encoder to freeze. This is what the organisers used
    #                       (2021), so it is the comparable control.
    #   w2v-BERT 2.0      — the SeamlessM4T encoder (2024, MIT). Consumes 80-bin
    #                       mel filterbanks, so a feature extractor sits in the
    #                       dataset. ~10x the pretraining audio of XLS-R, and
    #                       nothing in the organisers' table tried it.
    IS_BERT = "w2v-bert" in args.model.lower() or "bert" in args.model.lower()
    if IS_BERT:
        from transformers import (Wav2Vec2BertForCTC as CTCModel,
                                  AutoFeatureExtractor)
        feat_ex = AutoFeatureExtractor.from_pretrained(args.model)
    else:
        from transformers import Wav2Vec2ForCTC as CTCModel
        feat_ex = None

    dev = ("cuda" if torch.cuda.is_available() else
           "mps" if torch.backends.mps.is_available() else "cpu")

    def load_split(name):
        rows = []
        with open(f"{args.splits}/{name}.tsv", encoding="utf-8") as fh:
            for r in csv.DictReader(fh, delimiter="\t", quoting=csv.QUOTE_NONE):
                t = normalise(r["sentence"])
                if t:
                    rows.append({"path": r["path"], "text": t,
                                 "client_id": r.get("client_id", "")})
        return rows[: args.limit] if args.limit else rows

    tr, dv, te = load_split("train"), load_split("dev"), load_split("test")
    print(f"train {len(tr)}  dev {len(dv)}  test {len(te)}  |  {dev}", flush=True)

    trs = {r["text"] for r in tr}
    for n, s in (("dev", dv), ("test", te)):
        leak = sum(1 for r in s if r["text"] in trs)
        print(f"  sentence leakage {n}: {leak}/{len(s)} ({100*leak/max(len(s),1):.2f}%)")

    vocab = (json.load(open(args.vocab, encoding="utf-8"))
             if os.path.exists(args.vocab)
             else build_vocab([r["text"] for r in tr], args.vocab))
    inv = {v: k for k, v in vocab.items()}
    blank = vocab["[PAD]"]

    class LengthBucketed(torch.utils.data.Sampler):
        """Batch clips of similar length. Padding is charged at the longest
        item, so mixing a 3 s clip with a 20 s one wastes most of the compute."""
        def __init__(self, rows, batch, seed):
            import soundfile  # noqa: F401  (duration via header only)
            self.rows, self.batch, self.seed, self.epoch = rows, batch, seed, 0
            self.order = list(range(len(rows)))

        def set_lengths(self, lengths):
            self.order = sorted(range(len(lengths)), key=lambda i: lengths[i])

        def __iter__(self):
            batches = [self.order[i:i + self.batch]
                       for i in range(0, len(self.order), self.batch)]
            rng = random.Random(self.seed + self.epoch)
            rng.shuffle(batches)
            self.epoch += 1
            for b in batches:
                yield b

        def __len__(self):
            return (len(self.order) + self.batch - 1) // self.batch

    model = CTCModel.from_pretrained(
        args.model, vocab_size=len(vocab), ignore_mismatched_sizes=True,
        ctc_loss_reduction="mean", pad_token_id=blank,
        attention_dropout=0.0, hidden_dropout=0.0,
        feat_proj_dropout=0.0, layerdrop=0.0, mask_time_prob=0.05,
    )
    # The CNN feature encoder stays frozen: it encodes generic acoustics learnt
    # from far more speech than Quechua has, and 30 h is not enough to improve
    # it. Everything above it trains.
    if hasattr(model, "freeze_feature_encoder") and not IS_BERT:
        model.freeze_feature_encoder()
    model.gradient_checkpointing_enable()
    model.to(dev)
    n_tr = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"  trainable {n_tr/1e6:.0f}M of "
          f"{sum(p.numel() for p in model.parameters())/1e6:.0f}M", flush=True)

    dl = torch.utils.data.DataLoader(
        Utt(tr, args.clips, vocab, args.max_sec,
            augment=args.speed_perturb, feature_extractor=feat_ex), batch_size=args.batch,
        shuffle=True, collate_fn=collate, num_workers=args.workers, drop_last=True)
    steps = (len(dl) // args.accum) * args.epochs
    use_amp = dev == "cuda"
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
    opt = torch.optim.AdamW(
        [p for p in model.parameters() if p.requires_grad], lr=args.lr)
    sched = transformers.get_linear_schedule_with_warmup(
        opt, int(args.warmup * steps), steps)
    print(f"  {steps} optimiser steps over {args.epochs} epochs", flush=True)

    @torch.no_grad()
    def evaluate(rows, tag):
        model.eval()
        ce = cn = we = wn = 0
        loader = torch.utils.data.DataLoader(
            Utt(rows, args.clips, vocab, args.max_sec, feature_extractor=feat_ex),
            batch_size=args.batch, collate_fn=collate, num_workers=0)
        preds = []
        nb = len(loader)
        for bi, (X, _, xl, _, ids) in enumerate(loader):
            if bi % 25 == 0:
                print(f"    [{tag}] batch {bi}/{nb}", flush=True)
            with torch.autocast("cuda", dtype=torch.float16, enabled=use_amp):
                lg = model(X.to(dev),
                           attention_mask=(torch.arange(X.shape[1])[None, :]
                                           < xl[:, None]).long().to(dev)).logits
            lg = lg.float()
            out_len = model._get_feat_extract_output_lengths(xl).tolist()
            for k, seq in enumerate(lg.argmax(-1).cpu().tolist()):
                # CTC collapse: drop repeats, then drop blanks.
                prev, chars = None, []
                for t in seq[: out_len[k]]:
                    if t != prev and t != blank:
                        chars.append(inv.get(t, ""))
                    prev = t
                hyp = "".join(chars).replace("|", " ").strip()
                hyp = re.sub(r"\s+", " ", hyp)
                preds.append((ids[k], hyp))
        ref = {r["path"]: r["text"] for r in rows}
        for pid, hyp in preds:
            a, b = cer(ref[pid], hyp); ce += a; cn += b
            a, b = wer(ref[pid], hyp); we += a; wn += b
        model.train()
        return {"tag": tag, "cer": 100 * ce / cn, "wer": 100 * we / wn,
                "n": len(preds)}, preds

    os.makedirs(args.out, exist_ok=True)
    run = f"{args.out}/seed{args.seed}"
    os.makedirs(run, exist_ok=True)
    model.train()
    # The resume checkpoint is ephemeral: needed only inside a session, never
    # part of the output. /kaggle/temp is a separate filesystem that does NOT
    # count against the 19.5 GB /kaggle/working quota, and putting a 2-7 GB
    # checkpoint there is what stops it starving the next model. Falls back to
    # the run directory off-Kaggle.
    CKPT_DIR = "/kaggle/temp" if os.path.isdir("/kaggle/temp") else run
    os.makedirs(CKPT_DIR, exist_ok=True)
    CKPT = f"{CKPT_DIR}/{os.path.basename(args.out)}_seed{args.seed}_last.pt"
    print(f"  resume checkpoint -> {CKPT}", flush=True)
    import shutil as _sh
    for _d in ("/kaggle/working", CKPT_DIR):
        if os.path.isdir(_d):
            print(f"  free on {_d}: {_sh.disk_usage(_d).free/1e9:.1f} GB", flush=True)
    if args.hf_repo and not os.path.exists(CKPT):
        hf_pull(args.hf_repo, args.hf_token, f"{run}/best")
    step, start_ep, best = load_ckpt(CKPT, model, opt, sched, scaler)
    model.to(dev)
    if step:
        print(f"  skipping {step} completed steps", flush=True)
    t0 = time.time()
    for ep in range(start_ep, args.epochs):
        run_loss = k = 0
        # Steps already completed in a previous session are skipped without
        # a forward pass, so a resumed run does not redo work or re-consume
        # the schedule.
        seen = 0
        for i, (X, Y, xl, yl, _) in enumerate(dl):
            if seen < step - ep * (len(dl) // args.accum) * args.accum:
                seen += 1
                continue
            am = (torch.arange(X.shape[1])[None, :] < xl[:, None]).long()
            with torch.autocast("cuda", dtype=torch.float16, enabled=use_amp):
                out = model(X.to(dev), attention_mask=am.to(dev))
            # CTC loss stays in fp32 regardless: log_softmax over a long
            # sequence underflows in half precision and the loss goes to inf.
            lg = out.logits.float().log_softmax(-1).transpose(0, 1)
            loss = torch.nn.functional.ctc_loss(
                lg, Y.to(dev),
                model._get_feat_extract_output_lengths(xl).to(dev), yl.to(dev),
                blank=blank, reduction="mean", zero_infinity=True)
            scaler.scale(loss / args.accum).backward()
            run_loss += loss.item(); k += 1
            if (i + 1) % args.accum == 0:
                scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(opt); scaler.update()
                sched.step(); opt.zero_grad(set_to_none=True)
                step += 1
                if step % 50 == 0:
                    print(f"  ep{ep} step {step}/{steps} loss {run_loss/max(k,1):.3f} "
                          f"{(time.time()-t0)/60:.1f}m", flush=True)
                    run_loss = k = 0
                if args.save_every and step % args.save_every == 0:
                    save_ckpt(CKPT, model, opt, sched, scaler, step, ep, best, args)
        save_ckpt(CKPT, model, opt, sched, scaler, step, ep + 1, best, args)
        m, _ = evaluate(dv, "dev")
        print(f"[epoch {ep}] dev CER {m['cer']:.2f}  WER {m['wer']:.2f}", flush=True)
        if m["cer"] < best:
            best = m["cer"]
            save_ckpt(CKPT, model, opt, sched, scaler, step, ep + 1, best, args)
            model.save_pretrained(f"{run}/best")
            json.dump(vocab, open(f"{run}/best/vocab.json", "w"), ensure_ascii=False)
            json.dump({"epoch": ep, "step": step, "dev": m},
                      open(f"{run}/best/dev_metrics.json", "w"), indent=1)
            hf_push(f"{run}/best", args.hf_repo, args.hf_token,
                    f"epoch {ep} dev CER {m['cer']:.2f} WER {m['wer']:.2f}")

    m, preds = evaluate(te, "test")
    print(f"\n[TEST] CER {m['cer']:.2f}  WER {m['wer']:.2f}  n={m['n']}", flush=True)
    with open(f"{run}/predictions_test.tsv", "w", encoding="utf-8") as fh:
        for pid, hyp in preds:
            fh.write(f"{pid}\t{hyp}\n")
    json.dump({"test": m, "args": vars(args),
               "versions": {"transformers": transformers.__version__,
                            "torch": torch.__version__}},
              open(f"{run}/results.json", "w"), indent=1)
    print(f"wrote {run}/results.json")
    # The resume checkpoint is useless once the run is complete, and leaving it
    # behind is what starved the next model of disk. best/ keeps the weights.
    if os.path.exists(CKPT):
        os.remove(CKPT)
        print(f"removed {CKPT} (run complete)")


if __name__ == "__main__":
    main()
'''
SCRIPTS['eval_qxp.py']=r'''"""Score a saved checkpoint on a split. Standalone, so a training run that
stalls or is cancelled never costs you the number.

num_workers=0 on purpose: train_qxp.py built a fresh multi-worker DataLoader
every epoch and never tore them down, which is the likeliest cause of the
silent stall after epoch 7. Decoding is GPU-bound anyway.

    python eval_qxp.py --ckpt runs_xlsr/seed1/best --split test
"""
import argparse, csv, json, os, re, sys, time

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
from train_qxp import normalise, read_audio, cer, wer


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--ckpt", required=True)
    ap.add_argument("--splits", default="splits_grouped")
    ap.add_argument("--split", default="test")
    ap.add_argument("--clips", required=True)
    ap.add_argument("--vocab", default="vocab_qxp.json")
    ap.add_argument("--batch", type=int, default=4)
    ap.add_argument("--max-sec", type=float, default=20.0)
    ap.add_argument("--out", default=None)
    a = ap.parse_args()

    import torch
    from transformers import Wav2Vec2ForCTC, Wav2Vec2BertForCTC, AutoConfig

    vocab = json.load(open(a.vocab, encoding="utf-8"))
    inv = {v: k for k, v in vocab.items()}
    blank = vocab["[PAD]"]
    dev = "cuda" if torch.cuda.is_available() else "cpu"

    cfg = AutoConfig.from_pretrained(a.ckpt)
    is_bert = "bert" in cfg.model_type
    Model = Wav2Vec2BertForCTC if is_bert else Wav2Vec2ForCTC
    fe = None
    if is_bert:
        from transformers import AutoFeatureExtractor
        fe = AutoFeatureExtractor.from_pretrained("facebook/w2v-bert-2.0")
    model = Model.from_pretrained(a.ckpt).to(dev).eval()

    rows = []
    with open(f"{a.splits}/{a.split}.tsv", encoding="utf-8") as fh:
        for r in csv.DictReader(fh, delimiter="\t", quoting=csv.QUOTE_NONE):
            t = normalise(r["sentence"])
            if t:
                rows.append((r["path"], t))
    print(f"{len(rows)} clips from {a.split}  |  {dev}  |  {cfg.model_type}", flush=True)

    ce = cn = we = wn = 0
    preds = []
    t0 = time.time()
    for i in range(0, len(rows), a.batch):
        chunk = rows[i:i + a.batch]
        ys = [read_audio(a.clips, p, a.max_sec) for p, _ in chunk]
        if fe is not None:
            ys = [fe(y.numpy(), sampling_rate=16000,
                     return_tensors="pt").input_features[0] for y in ys]
        xl = torch.tensor([len(y) for y in ys])
        if ys[0].dim() == 2:
            X = torch.zeros(len(ys), int(max(xl)), ys[0].shape[1])
        else:
            X = torch.zeros(len(ys), int(max(xl)))
        for k, y in enumerate(ys):
            X[k, : len(y)] = y
        am = (torch.arange(X.shape[1])[None, :] < xl[:, None]).long()
        with torch.no_grad():
            lg = model(X.to(dev), attention_mask=am.to(dev)).logits.float()
        out_len = model._get_feat_extract_output_lengths(xl).tolist()
        for k, seq in enumerate(lg.argmax(-1).cpu().tolist()):
            prev, chars = None, []
            for t in seq[: out_len[k]]:
                if t != prev and t != blank:
                    chars.append(inv.get(t, ""))
                prev = t
            hyp = re.sub(r"\s+", " ", "".join(chars).replace("|", " ")).strip()
            ref = chunk[k][1]
            preds.append((chunk[k][0], ref, hyp))
            x, y = cer(ref, hyp); ce += x; cn += y
            x, y = wer(ref, hyp); we += x; wn += y
        if (i // a.batch) % 25 == 0:
            done = min(i + a.batch, len(rows))
            print(f"  {done}/{len(rows)}  running CER {100*ce/max(cn,1):.2f}  "
                  f"WER {100*we/max(wn,1):.2f}  {(time.time()-t0)/60:.1f}m", flush=True)

    m = {"split": a.split, "ckpt": a.ckpt, "n": len(preds),
         "cer": 100 * ce / cn, "wer": 100 * we / wn}
    print(f"\n[{a.split.upper()}] CER {m['cer']:.2f}  WER {m['wer']:.2f}  n={m['n']}")
    out = a.out or f"eval_{a.split}.json"
    json.dump(m, open(out, "w"), indent=1)
    with open(out.replace(".json", ".tsv"), "w", encoding="utf-8") as fh:
        for p, ref, hyp in preds:
            fh.write(f"{p}\t{ref}\t{hyp}\n")
    print("wrote", out)


if __name__ == "__main__":
    main()
'''
for fn,src in SCRIPTS.items():
    open(fn,'w',encoding='utf-8').write(src); print(f'wrote {fn} ({len(src.splitlines())} lines)')

In [ ]:
!python -u make_split_grouped.py --input "{QXP}/validated.tsv" --output-dir splits_grouped

## XLS-R 300M

About 2.8 h. Re-running this cell resumes, from Hugging Face if the container
was replaced.


In [ ]:
%%time
!python -u train_qxp.py --model facebook/wav2vec2-xls-r-300m \
    --splits splits_grouped --clips "{CLIPS}" --vocab vocab_qxp.json \
    --epochs 6 --batch 8 --accum 4 --speed-perturb \
    --save-every 100 --workers 2 --hf-repo {HF_USER}/qxp-xlsr \
    --out runs_xlsr 2>&1 | tee -a train_xlsr.log

In [ ]:
!python -u eval_qxp.py --ckpt runs_xlsr/seed1/best --split test \
    --clips "{CLIPS}" --vocab vocab_qxp.json --splits splits_grouped \
    --out eval_xlsr_test.json 2>&1 | tee -a eval_xlsr.log

## Progress


In [ ]:
!tail -30 train_xlsr.log 2>/dev/null; echo
!echo "--- checkpoints ---"; du -sh runs_*/seed1/* 2>/dev/null
!echo "--- disk ---"; df -h /kaggle/working | tail -1
!echo "--- gpu ---"; nvidia-smi --query-gpu=utilization.gpu,memory.used --format=csv,noheader

## Free disk before the 606M model


In [ ]:
import shutil, os
for p in ['runs_xlsr/seed1/last.pt','runs_xlsr/seed1/last.pt.tmp']:
    if os.path.exists(p): os.remove(p); print('removed',p)
!rm -rf /root/.cache/huggingface/hub/models--facebook--wav2vec2-xls-r-300m
free=shutil.disk_usage("/kaggle/working").free/1e9
print(f'free now: {free:.1f} GB')
assert free > 6, "under 6 GB free — clear more before starting w2v-BERT"

## w2v-BERT 2.0

About 3–4 h.


In [ ]:
%%time
!python -u train_qxp.py --model facebook/w2v-bert-2.0 \
    --splits splits_grouped --clips "{CLIPS}" --vocab vocab_qxp.json \
    --epochs 6 --batch 4 --accum 8 --speed-perturb \
    --save-every 200 --workers 2 --hf-repo {HF_USER}/qxp-w2vbert \
    --out runs_w2vbert 2>&1 | tee -a train_w2vbert.log

In [ ]:
!python -u eval_qxp.py --ckpt runs_w2vbert/seed1/best --split test \
    --clips "{CLIPS}" --vocab vocab_qxp.json --splits splits_grouped \
    --out eval_w2vbert_test.json 2>&1 | tee -a eval_w2vbert.log

In [ ]:
import json, os
print(f'{"model":30s} {"CER":>7s} {"WER":>7s}   n')
print('-'*56)
for tag,p in [('xls-r-300m (2021, control)','eval_xlsr_test.json'),
              ('w2v-bert-2.0 (2024)','eval_w2vbert_test.json')]:
    if os.path.exists(p):
        m=json.load(open(p)); print(f'{tag:30s} {m["cer"]:7.2f} {m["wer"]:7.2f}   {m["n"]}')
    else: print(f'{tag:30s}   (not run)')
print('\nOrganisers, scripted, THEIR split: 1.19 WER / 0.19 CER')
print('  -> not comparable: 99.96% of their test sentences are in their train text.')